<a href="https://colab.research.google.com/github/nikitask14/pytorch-engineering-to-federated-learning/blob/main/17_mnist_dataset_and_dataloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import torch
from torchvision import datasets, transforms
import torch.nn as nn

torch is a core pytorch package used for tensors, autograd, neural netwrok computation, optimisers etc.

torchvision is a companion package containing tools commonly needed for computervision data.

In [4]:
train_dataset = datasets.MNIST(
    root = "data",
    train = True,
    download = True,
    transform=transforms.ToTensor()
)

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 491kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.35MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.24MB/s]


Use the MNIST class available inside datasets. Store or look for the MNIST data in a folder called data. Give me the training split. If the MNIST data files are not already there, download them. When an image is retrieved, convert it to a PyTorch tensor.

In [5]:
image, label = train_dataset[0]
print(image.shape)
print(label)
print(type(image))
print(type(label))

torch.Size([1, 28, 28])
5
<class 'torch.Tensor'>
<class 'int'>


Inspecting one MNIST example

A single item from train_dataset contains two things:


*   an image
*   its corresponding label


We can retrieve the first example using:

##$$image, label = train__dataset[0]$$

train_dataset[0] returns the first (image, label) pair, and Python unpacking stores the two returned values separately.

For this example:

1. image is a PyTorch tensor.
2. Its shape is (1, 28, 28), which follows the format (channel, height, width).
3. 1 represents the single grayscale channel.
4. 28 × 28 represents the height and width of the image.
5. Therefore, the image contains 28 × 28 = 784 pixel-intensity values.
label is a Python integer representing the correct digit class, from 0 to 9.

Unlike the image, the individual label is not a tensor at this stage, so it does not have a .shape attribute.

In [6]:
print(image.min())
print(image.max())

tensor(0.)
tensor(1.)


So in this particular MNIST image:

1. the minimum pixel intensity is 0.0
2. the maximum pixel intensity is 1.0

That verifies that transforms.ToTensor() has converted the original image values onto a floating-point 0–1 scale.

So ToTensor() does two useful things for MNIST:

original image

integer pixel values
0 ... 255 -> ToTensor() -> floating-point tensor (0.0 ... 1.0)

For example:

$$ 0 \rightarrow 0.0 $$ $$ 127 \rightarrow \frac{127}{255}\approx 0.498 $$ $$ 255 \rightarrow 1.0 $$

Notice that the information about brightness has not changed. A brighter pixel is still brighter than a darker pixel. We have just changed the numerical representation.

Why also scale from 0–255 to 0–1?

Because neural networks generally train more comfortably when their inputs are on a smaller numerical scale.


---


IMPORTANT: Pixel intensities are already numbers, but they are stored as integers for image storage. ToTensor() converts them into floating-point values because neural-network computation and gradient-based learning operate on floating-point tensors, and it scales MNIST's 0–255 values to 0–1.

In [7]:
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size= 32, shuffle = True)

for image, label in train_loader:
  print(image.shape)
  print(label.shape)
  break

torch.Size([32, 1, 28, 28])
torch.Size([32])


Read it as:

*   32 images
*   1 channel per image
*   28 pixels high
*   28 pixels wide

So each individual image still has its 2D image structure:

(1, 28, 28)

In [10]:
flat_images = torch.flatten(image, start_dim = 1)

flatten means combine multiple dimesnsions into a single dimension.

start_dim = 1 means leave dimension 0 alone and flatten everything starting from dimension 1.

Important thing to note is that if we flattened even the 0th dimesnsion then it woud have mixed all images together,

In [12]:
flat_images.shape

torch.Size([32, 784])

In [16]:
class MyModel(nn.Module):
  def __init__(self):
    super().__init__()

    self.layer1 = nn.Linear(784,128)
    self.layer2 = nn.Linear(128,10)

  def forward(self, x):
    x = torch.relu(self.layer1(x))
    x = self.layer2(x)
    return x



In [22]:
model = MyModel()
loss_fn = nn.CrossEntropyLoss()
optimiser = torch.optim.SGD(model.parameters(), lr = 0.01)

In [23]:

for epoch in range(5):
  running_loss = 0.0
  model.train()
  for image, label in train_loader:
    flat_images = torch.flatten(image, start_dim = 1)
    optimiser.zero_grad()
    train_logits = model(flat_images)
    train_loss = loss_fn(train_logits, label)
    train_loss.backward()
    optimiser.step()
    running_loss +=train_loss.item()

  epoch_loss = running_loss/len(train_loader)


  print(
      f"Epoch:{epoch+1d}, Epoch Loss:{epoch_loss}"
       )



Epoch:0, Epoch Loss:0.8826413490613302
Epoch:1, Epoch Loss:0.3806912973046303
Epoch:2, Epoch Loss:0.32582581953605017
Epoch:3, Epoch Loss:0.2967577513595422
Epoch:4, Epoch Loss:0.27440427388946215
